# Labelling Pipeline

This notebook implements the labelling strategy used to build training data for our NLP benchmark classifier. The pipeline operates in three phases:

1. **RegEx bucketing** — Heuristic scoring assigns each paper to one of three confidence buckets (A, B, C) based on lexical signals in the title and abstract.
2. **Stratified sampling** — Papers are drawn from each bucket to form balanced gold, train, and validation splits.
3. **LLM labelling** — Two LLMs (Mistral Small, Claude Haiku) independently classify sampled papers; inter-annotator agreement is measured with Cohen's $\kappa$.

## 1. Configuration & Data Loading

In [ ]:
import sys
from pathlib import Path

import pandas as pd
from sklearn.metrics import cohen_kappa_score, confusion_matrix

sys.path.insert(0, str(Path().resolve().parent))

In [ ]:
DATA_DIR = Path("../data")

# Sampling parameters
SEED = 42
GOLD_PER_BUCKET = 70
TRAIN_N_A, TRAIN_N_B, TRAIN_N_C = 500, 400, 300
VAL_FRAC = 0.15

In [ ]:
anthology_df = pd.read_parquet(DATA_DIR / "raw" / "anthology_enriched.parquet")
print(f"Anthology corpus: {len(anthology_df):,} papers")
anthology_df[["bibkey", "title", "year"]].head()

## 2. RegEx Bucketing

Each paper is scored against hand-crafted regex patterns that capture lexical signals for benchmark introduction and text classification relevance. Signals are grouped into categories (e.g. `introduces`, `evaluates_on`, `classif_tasks`, `known_classif_benchmarks`) and aggregated into composite scores.

Papers are then assigned to one of three buckets:
- **Bucket A** — Strong evidence the paper introduces a text-classification benchmark (high-confidence positive).
- **Bucket B** — Partial or ambiguous evidence; requires LLM adjudication.
- **Bucket C** — No relevant signal (likely negative).

A post-filter detects papers that *introduce a method* (not a benchmark) to reduce false positives in Bucket A.

In [ ]:
from src.labelling.regex_buckets import assign_bucket, compute_scores

# Score every paper against regex signal categories
score_cols = anthology_df.apply(
    lambda r: compute_scores(r["title"], r["abstract"]), axis=1
)
score_df = pd.DataFrame(score_cols.tolist())

anthology_bucket_df = pd.concat(
    [anthology_df.reset_index(drop=True), score_df.reset_index(drop=True)],
    axis=1,
)
anthology_bucket_df["bucket"] = anthology_bucket_df.apply(assign_bucket, axis=1)

In [ ]:
bucket_counts = anthology_bucket_df["bucket"].value_counts().sort_index()
print("Bucket distribution:")
print(bucket_counts.to_string())
print(f"\nTotal: {len(anthology_bucket_df):,} papers")

In [ ]:
anthology_bucket_df.to_parquet(
    DATA_DIR / "labeled" / "anthology_enriched_with_bucket.parquet"
)

## 3. Stratified Sampling

We draw three non-overlapping splits from the bucketed corpus, each stratified by bucket and year to ensure temporal representativeness:

| Split | Purpose | Size per bucket |
|-------|---------|-----------------|
| **Gold** | Human-evaluated held-out test set | 70 / bucket |
| **Train** | LLM-labelled training data | A=500, B=400, C=300 |
| **Val** | Validation (15% of train pool) | ~15% of train per bucket |

In [ ]:
from src.labelling.regex_buckets import stratified_bucket_sample

# Gold set — held out for human evaluation
gold = stratified_bucket_sample(
    anthology_bucket_df,
    n_a=GOLD_PER_BUCKET,
    n_b=GOLD_PER_BUCKET,
    n_c=GOLD_PER_BUCKET,
    seed=SEED,
)
remaining = anthology_bucket_df.drop(gold.index)

# Train + Val pool
train_val = stratified_bucket_sample(
    remaining,
    n_a=TRAIN_N_A,
    n_b=TRAIN_N_B,
    n_c=TRAIN_N_C,
    seed=SEED + 1,
)

# Split train/val within each bucket
val_idx = (
    train_val.groupby("bucket", group_keys=False)
    .apply(lambda x: x.sample(frac=VAL_FRAC, random_state=SEED))
    .index
)
val = train_val.loc[val_idx]
train = train_val.drop(val_idx)

In [ ]:
summary = pd.DataFrame(
    {
        "Gold": gold["bucket"].value_counts(),
        "Train": train["bucket"].value_counts(),
        "Val": val["bucket"].value_counts(),
    }
).fillna(0).astype(int).sort_index()
summary.loc["Total"] = summary.sum()
summary

In [ ]:
splits_dir = DATA_DIR / "splits"
splits_dir.mkdir(parents=True, exist_ok=True)

gold.to_parquet(splits_dir / "gold.parquet", index=False)
train.to_parquet(splits_dir / "train.parquet", index=False)
val.to_parquet(splits_dir / "val.parquet", index=False)

## 4. LLM Labelling

Each paper in the training split is classified independently by two LLMs via their respective Batch APIs:
- **Mistral Small** (`mistral-small-latest`) — structured output via JSON schema.
- **Claude Haiku** (`claude-haiku-4-5-20251001`) — structured output via tool use.

Both models receive the same system prompt that defines POSITIVE / NEGATIVE / UNSURE labels (see `src/labelling/llm_labeller.py` for the full prompt and Pydantic schema).

> **Note:** Batch API calls are non-deterministic and cost-incurring. The cells below load pre-computed results. To re-run the batch jobs from scratch, uncomment the submission cells.

In [ ]:
from src.labelling.llm_labeller import (
    mistral_batch_submit,
    mistral_batch_results,
    claude_batch_submit,
    claude_batch_results,
)

### 4.1 Submit batch jobs (optional)

Uncomment the cells below to submit new batch jobs. Each call returns a job ID that can be used to retrieve results once processing completes.

In [ ]:
# job_id_mistral = mistral_batch_submit(train)
# job_id_mistral

In [ ]:
# job_id_claude = claude_batch_submit(train)
# job_id_claude

### 4.2 Load pre-computed results

In [ ]:
result_mistral = pd.read_parquet(DATA_DIR / "splits" / "train_mistral_1.parquet")
result_claude = pd.read_parquet(DATA_DIR / "splits" / "train_claude_1.parquet")

print(f"Mistral labels: {len(result_mistral):,} papers")
print(result_mistral["mistral_label"].value_counts().to_string())
print(f"\nClaude labels:  {len(result_claude):,} papers")
print(result_claude["claude_label"].value_counts().to_string())

## 5. Inter-Annotator Agreement

We measure agreement between the two LLM annotators using Cohen's $\kappa$, which corrects for chance agreement. Interpretation thresholds (Landis & Koch, 1977):

| $\kappa$ | Interpretation |
|-----------|----------------|
| < 0.20 | Slight |
| 0.21–0.40 | Fair |
| 0.41–0.60 | Moderate |
| 0.61–0.80 | Substantial |
| 0.81–1.00 | Almost perfect |

In [ ]:
# Align on shared bibkeys
merged = result_mistral[["bibkey", "mistral_label"]].merge(
    result_claude[["bibkey", "claude_label"]],
    on="bibkey",
    how="inner",
)
print(f"Papers with both labels: {len(merged):,}")

kappa = cohen_kappa_score(merged["mistral_label"], merged["claude_label"])
print(f"Cohen's kappa: {kappa:.3f}")

In [ ]:
labels = sorted(merged["mistral_label"].unique())
cm = confusion_matrix(merged["mistral_label"], merged["claude_label"], labels=labels)
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
cm_df.index.name = "Mistral"
cm_df.columns.name = "Claude"
cm_df

### Disagreement analysis

Papers where the two annotators disagree are the most informative cases for understanding model biases and refining the classification prompt.

In [ ]:
disagreements = merged[merged["mistral_label"] != merged["claude_label"]]
print(f"Disagreements: {len(disagreements)} / {len(merged)} ({len(disagreements)/len(merged):.1%})")

# Show disagreement breakdown
pd.crosstab(
    disagreements["mistral_label"],
    disagreements["claude_label"],
    margins=True,
)